In [1]:
!pip install --upgrade 'pydantic>=2.0,<3.0' openai anthropic google-generativeai together pandas numpy --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.15.1 requires numpy<2.0.0,>=1.23.5, but you have numpy 2.2.6 which is incompatible.
spacy 3.4.4 requires pydantic!=1.8,!=1.8.1,<1.11.0,>=1.7.4, but you have pydantic 2.12.4 which is incompatible.
spacy 3.4.4 requires typer<0.8.0,>=0.3.0, but you have typer 0.19.2 which is incompatible.
scipy 1.9.3 requires numpy<1.26.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
deepnote-toolkit 1.1.5 requires matplotlib-inline<0.2.0,>=0.1.7; python_version <= "3.10", but you have matplotlib-inline 0.2.1 which is incompatible.
deepnote-toolkit 1.1.5 requires numpy<2,>=1.23; python_version == "3.10", but you have numpy 2.2.6 which is incompatible.
deepnote-toolkit 1.1.5 requires pandas<2.2,>=1.2.5; python_version < "3.12", but you have pandas 2.3.3 which is incompatible.

[notice] A new release of pip is 

In [1]:
#!/usr/bin/env python3
"""
LLM INFERENCE SCRIPT - HARDCODED PROMPTS FROM PDF (FIXED)
=================================================================
This script uses the EXACT 24 prompts from the uploaded PDF to ensure
100% fidelity to the original study design.

FIXED VERSION:
- Each row in the CSV corresponds to ONE question/PI
- Predictions are assigned to the CORRECT row based on question number
- Column names: gpt_predicted, claude_predicted, gemini_predicted, llama_predicted
- Order matches the CSV file (using 'qnum' column to map questions)
"""

import pandas as pd
import openai
import anthropic
import google.generativeai as genai
from together import Together
import time
import os
from typing import Dict, Optional

# ================================================================
# Configuration
# ================================================================

# System instruction to prevent academic citations and data leakage
SYSTEM_INSTRUCTION = """You are a prediction assistant making estimates based ONLY on the information provided in this specific prompt.

CRITICAL INSTRUCTIONS:
1. Do NOT cite, reference, or mention ANY research papers, academic studies, surveys, or authors (including but not limited to Sparkman et al., Leviston et al., Lees et al., or any other researchers)
2. Do NOT use any memorized data, statistics, or percentages from your training about climate change opinions, pluralistic ignorance, or survey results
3. Treat this as a completely NOVEL scenario - ignore any similar studies you may have seen during training
4. Do NOT reference 'research shows', 'studies indicate', 'surveys have found', or similar phrases
5. Base your estimate ONLY on:
   - General reasoning about human psychology and behavior
   - The specific information provided in this prompt
   - First principles about how people form beliefs about others

Your task is to predict what percentage people THINK others believe (second-order belief), not what people actually believe (first-order belief). This is a prediction task requiring general reasoning, not recall of specific research findings.

Respond with ONLY a JSON object containing a single number between 0 and 100 with one decimal place: {"prediction": XX.X}

Do not include any explanation, reasoning, or text - only the JSON."""

# API keys (set these as environment variables)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CLAUDE_API_KEY = os.getenv("CLAUDE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")  # Google AI Studio API key (for Gemini)
LLAMA_API_KEY = os.getenv("LLAMA_API_KEY")  # Together AI (for Llama)

# Model configurations
MODELS = {
    "gpt": "gpt-4o-mini",
    "claude": "claude-3-5-haiku-20241022",
    "gemini": "gemini-2.5-flash",
    "llama": "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"
}

# API clients
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
anthropic_client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
genai.configure(api_key=GEMINI_API_KEY)
together_client = Together(api_key=LLAMA_API_KEY)

# ================================================================
# All 24 Hardcoded Prompts (Exact from PDF)
# ================================================================

PROMPTS = {
1: """In 2021, in a nationally representative survey with a sample of 6119 adults aged 18 years and above in the United States, respondents were asked:

"How worried are you about climate change?"

Responses: Very worried, somewhat worried, not very worried, not at all worried

Respondents were then asked what percentage of Americans feel at least "somewhat" worried about climate change.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER respondents in the United States feel at least “somewhat” worried about climate change.

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",


2: """In 2021, in a nationally representative survey with a sample of 6119 adults aged 18 years and above in the United States, respondents were asked:

"How much do you support or oppose the following policy: require fossil fuel companies to pay a carbon tax and use the money to reduce other taxes (such as income tax) by an equal amount?"

Responses: Strongly support, somewhat support, somewhat oppose, strongly oppose.

Respondents were then asked what percentage of Americans support requiring fossil fuel companies to pay a carbon tax and use the money to reduce other taxes (such as income tax) by an equal amount.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER respondents in the United States support requiring fossil fuel companies to pay a carbon tax and use the money to reduce other taxes (such as income tax) by an equal amount.

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

3: """In 2021, in a nationally representative survey with a sample of 6119 adults aged 18 years and above in the United States, respondents were asked:

"How much do you support or oppose the following policy: require electric utilities to produce 100% of their electricity from renewable energy sources by the year 2035?"

Responses: Strongly support, somewhat support, somewhat oppose, strongly oppose.

Respondents were then asked what percentage of Americans support requiring electric utilities to produce 100% of their electricity from renewable energy sources by the year 2035.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER respondents in the United States support requiring electric utilities to produce 100% of their electricity from renewable energy sources by the year 2035.

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

4: """In 2021, in a nationally representative survey with a sample of 6119 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: generate renewable energy (solar and wind) on public land in the U.S.?”

Responses: Strongly support, somewhat support, somewhat oppose, strongly oppose.

Respondents were then asked what percentage of Americans support generating renewable energy (solar and wind) on public land in the U.S.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER respondents in the United States support generating renewable energy (solar and wind) on public land in the U.S.

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

5: """In 2021, in a nationally representative survey with a sample of 6119 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: Green New Deal to produce jobs and strengthen America’s economy by accelerating the transition from fossil fuels to clean, renewable energy? The "Deal" would generate 100% of the nation's electricity from clean sources within 10 years, upgrade the U.S. energy grid, buildings, and transportation infrastructure, increase energy efficiency, invest in green-technology R&D, and provide training for jobs in the new green economy.”

Responses: Strongly support, somewhat support, somewhat oppose, strongly oppose.

Respondents were then asked what percentage of Americans support the Green New Deal to produce jobs and strengthen America’s economy by accelerating the transition from fossil fuels to clean, renewable energy.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER respondents in the United States support the Green New Deal to produce jobs and strengthen America’s economy by accelerating the transition from fossil fuels to clean, renewable energy.

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",





6: """In 2025, in a nationally representative probability-based survey with a sample of 5068 adults aged 18 years and above in the United States, respondents were asked:

“How much of a problem do you think climate change is in the country today?”

Responses: A very big problem, A moderately big problem, A small problem, Not a problem at all, and No answer. 

Respondents were then asked what percentage of Americans indicate that climate change is a very big problem in the country today.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER respondents in the United States indicate that climate change is a very big problem in the country today.

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

7: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: regulate carbon dioxide (the primary greenhouse gas) as a pollutant?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly support or somewhat support regulating carbon dioxide (the primary greenhouse gas) as a pollutant.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly support or somewhat support regulating carbon dioxide (the primary greenhouse gas) as a pollutant.

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",
8: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: require electric utilities to produce 100% of their electricity from wind, solar, or other renewable energy sources by the year 2035?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly support or somewhat support requiring electric utilities to produce 100% of their electricity from wind, solar, or other renewable energy sources by the year 2035.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly support or somewhat support requiring electric utilities to produce 100% of their electricity from wind, solar, or other renewable energy sources by the year 2035. 

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

9: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: fund more research into renewable energy sources, such as solar and wind power?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly support or somewhat support funding more research into renewable energy sources, such as solar and wind power.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly support or somewhat support funding more research into renewable energy sources, such as solar and wind power. 

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

10: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: provide tax rebates for people who purchase energy-efficient vehicles or solar panels?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly support or somewhat support providing tax rebates for people who purchase energy-efficient vehicles or solar panels.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly support or somewhat support providing tax rebates for people who purchase energy-efficient vehicles or solar panels. 

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

11: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: require fossil fuel companies to pay a carbon tax and use the money to reduce other taxes (such as income tax) by an equal amount?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly support or somewhat support requiring fossil fuel companies to pay a carbon tax and use the money to reduce other taxes (such as income tax) by an equal amount.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly support or somewhat support requiring fossil fuel companies to pay a carbon tax and use the money to reduce other taxes (such as income tax) by an equal amount. 

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

12: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: transition the U.S. economy (incl. electric utilities, transportation, buildings, industry) from fossil fuels to 100% clean energy by 2050?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly support or somewhat support transitioning the U.S. economy (incl. electric utilities, transportation, buildings, industry) from fossil fuels to 100% clean energy by 2050.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly support or somewhat support transitioning the U.S. economy (incl. electric utilities, transportation, buildings, industry) from fossil fuels to 100% clean energy by 2050. 

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",




13: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: provide tax incentives or rebates to homeowners, landlords, and businesses to purchase appliances that can be powered without burning fossil fuels?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly support or somewhat support providing tax incentives or rebates to homeowners, landlords, and businesses to purchase appliances that can be powered without burning fossil fuels.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly support or somewhat support providing tax incentives or rebates to homeowners, landlords, and businesses to purchase appliances that can be powered without burning fossil fuels. 

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

14: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: provide tax incentives or rebates to homeowners, landlords, and businesses to make existing buildings more energy efficient?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly support or somewhat support providing tax incentives or rebates to homeowners, landlords, and businesses to make existing buildings more energy efficient.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly support or somewhat support providing tax incentives or rebates to homeowners, landlords, and businesses to make existing buildings more energy efficient. 

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

15: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: provide federal funding to make residential buildings in low-income communities more energy efficient?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly support or somewhat support providing federal funding to make residential buildings in low-income communities more energy efficient.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly support or somewhat support providing federal funding to make residential buildings in low-income communities more energy efficient. 

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

16: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: increase federal funding to low-income communities and communities of color who are disproportionately harmed by air and water pollution?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly support or somewhat support increasing federal funding to low-income communities and communities of color who are disproportionately harmed by air and water pollution.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly support or somewhat support increasing federal funding to low-income communities and communities of color who are disproportionately harmed by air and water pollution. 

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

17: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: expand offshore drilling for oil and natural gas off the U.S. coast?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly oppose or somewhat oppose expanding offshore drilling for oil and natural gas off the U.S. coast.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly oppose or somewhat oppose expanding offshore drilling for oil and natural gas off the U.S. coast.

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",


18: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: drill for and mine fossil fuels (coal, oil, and natural gas) on public land in the U.S.?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly oppose or somewhat oppose drilling for and mining fossil fuels (coal, oil, and natural gas) on public land in the U.S.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly oppose or somewhat oppose drilling for and mining fossil fuels (coal, oil, and natural gas) on public land in the U.S.

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

19: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: generate renewable energy (solar and wind) on public land in the U.S.?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly support or somewhat support generating renewable energy (solar and wind) on public land in the U.S.

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly support or somewhat support generating renewable energy (solar and wind) on public land in the U.S.

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

20: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: re-establish the Civilian Conservation Corps, which would employ workers to protect natural ecosystems, plant trees in rural and urban areas, and restore the soil on farmlands?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly support or somewhat support re-establishing the Civilian Conservation Corps, which would employ workers to protect natural ecosystems, plant trees in rural and urban areas, and restore the soil on farmlands. 

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly support or somewhat support re-establishing the Civilian Conservation Corps, which would employ workers to protect natural ecosystems, plant trees in rural and urban areas, and restore the soil on farmlands.

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

21: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: create a jobs program that would hire unemployed oil and gas workers to safely close down thousands of abandoned oil and gas wells, which are a source of water and methane pollution?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly support or somewhat support creating a jobs program that would hire unemployed oil and gas workers to safely close down thousands of abandoned oil and gas wells, which are a source of water and methane pollution. 

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly support or somewhat support creating a jobs program that would hire unemployed oil and gas workers to safely close down thousands of abandoned oil and gas wells, which are a source of water and methane pollution.

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

22: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: create a jobs program that would hire unemployed coal workers to safely close down old coal mines and restore the natural landscape?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly support or somewhat support creating a jobs program that would hire unemployed coal workers to safely close down old coal mines and restore the natural landscape. 

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly support or somewhat support creating a jobs program that would hire unemployed coal workers to safely close down old coal mines and restore the natural landscape.

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

23: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“How much do you support or oppose the following policy: provide federal funding to help farmers improve farming practices to protect and restore the soil so it absorbs and stores more carbon?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly support or somewhat support providing federal funding to help farmers improve farming practices to protect and restore the soil so it absorbs and stores more carbon. 

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly support or somewhat support providing federal funding to help farmers improve farming practices to protect and restore the soil so it absorbs and stores more carbon.

Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place.""",

24: """In 2022, in a nationally representative non-probability survey with a sample of 916 adults aged 18 years and above in the United States, respondents were asked:

“If Congress does not take action on global warming, would you support or oppose a U.S. president declaring global warming a national emergency to act on it without Congress?”

Responses: Strongly support, Somewhat support, Somewhat oppose, Strongly oppose, and No answer.

Respondents were then asked what percentage of all registered voters in the United States they think strongly support or somewhat support a U.S. president declaring global warming a national emergency to act on it without Congress if Congress does not take action on global warming. 

Responses: between 0% and 100%. 

Based on the country, estimate what respondents in the United States, on average thought about how many OTHER registered voters in the United States strongly support or somewhat support a U.S. president declaring global warming a national emergency to act on it without Congress if Congress does not take action on global warming. 
 
Note: You are estimating people's BELIEFS about others’ beliefs NOT their own personal views.

Respond with a single number between 0 and 100 with one decimal place."""
}

# ================================================================
# LLM API Functions
# ================================================================

def call_gpt(prompt: str, question_num: int) -> Optional[float]:
    """Call GPT-4o-mini API"""
    try:
        response = openai_client.chat.completions.create(
            model=MODELS["gpt"],
            messages=[
                {"role": "system", "content": SYSTEM_INSTRUCTION},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=50
        )
        result = response.choices[0].message.content.strip()
        return parse_prediction(result, "gpt", question_num)
    except Exception as e:
        print(f"  ❌ GPT Error Q{question_num}: {e}")
        return None

def call_claude(prompt: str, question_num: int) -> Optional[float]:
    """Call Claude-3.5-Haiku API"""
    try:
        response = anthropic_client.messages.create(
            model=MODELS["claude"],
            max_tokens=50,
            temperature=0.0,
            system=SYSTEM_INSTRUCTION,
            messages=[{"role": "user", "content": prompt}]
        )
        result = response.content[0].text.strip()
        return parse_prediction(result, "claude", question_num)
    except Exception as e:
        print(f"  ❌ Claude Error Q{question_num}: {e}")
        return None

def call_gemini(prompt: str, question_num: int) -> Optional[float]:
    """Call Gemini-2.5-Flash API"""
    try:
        genai.configure(api_key=GEMINI_API_KEY)
        model = genai.GenerativeModel(MODELS["gemini"])
        
        # Combine system instruction with prompt (same as ablation script)
        full_prompt = f"{SYSTEM_INSTRUCTION}\n\n{prompt}"
        
        response = model.generate_content(
            full_prompt,
            generation_config=genai.types.GenerationConfig(
                temperature=0.0,
                max_output_tokens=50
            )
        )
        
        result = response.text.strip()
        return parse_prediction(result, "gemini", question_num)
    except Exception as e:
        print(f"  ❌ Gemini Error Q{question_num}: {e}")
        return None

def call_llama(prompt: str, question_num: int) -> Optional[float]:
    """Call Llama-4-Maverick API via Together"""
    try:
        response = together_client.chat.completions.create(
            model=MODELS["llama"],
            messages=[
                {"role": "system", "content": SYSTEM_INSTRUCTION},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=50
        )
        result = response.choices[0].message.content.strip()
        return parse_prediction(result, "llama", question_num)
    except Exception as e:
        print(f"  ❌ Llama Error Q{question_num}: {e}")
        return None

def parse_prediction(text: str, model_name: str, question_num: int) -> Optional[float]:
    """Extract numeric prediction from model response (handles both JSON and plain text)"""
    import re
    import json
    
    # Try to parse as JSON first
    try:
        # Remove markdown code blocks if present
        cleaned = text.replace('```json', '').replace('```', '').strip()
        data = json.loads(cleaned)
        if 'prediction' in data:
            value = float(data['prediction'])
            if 0 <= value <= 100:
                print(f"  ✓ {model_name.upper()} Q{question_num}: {value:.1f}%")
                return round(value, 1)
    except (json.JSONDecodeError, KeyError, ValueError):
        pass
    
    # Fallback: extract any number from text
    numbers = re.findall(r'\d+\.?\d*', text)
    
    if numbers:
        try:
            value = float(numbers[0])
            if 0 <= value <= 100:
                print(f"  ✓ {model_name.upper()} Q{question_num}: {value:.1f}%")
                return round(value, 1)
        except:
            pass
    
    print(f"  ⚠️  {model_name.upper()} Q{question_num}: Could not parse '{text}'")
    return None

# ================================================================
# Main Execution
# ================================================================

def main():
    """Run inference for all 24 questions"""
    
    print("="*100)
    print("LLM INFERENCE - HARDCODED PROMPTS FROM PDF (FIXED VERSION)")
    print("="*100)
    
    # Load the CSV
    df = pd.read_csv('pi_effects_llms.csv', encoding='latin-1')
    
    print(f"\n✓ Loaded {len(df)} rows from CSV")
    
    # Check if 'qnum' column exists to map questions to rows
    if 'qnum' not in df.columns:
        print("\n⚠️  WARNING: 'qnum' column not found in CSV!")
        print("   Assuming rows are in order (row 0 = Q1, row 1 = Q2, etc.)")
        df['qnum'] = range(1, len(df) + 1)
    
    print(f"✓ Using models: {list(MODELS.keys())}")
    print(f"✓ Hardcoded prompts: {len(PROMPTS)}")
    
    # Verify we have 24 rows
    if len(df) != 24:
        print(f"\n⚠️  WARNING: Expected 24 rows but found {len(df)} rows!")
        print("   Script will only process rows with qnum 1-24")
    
    # Initialize result columns
    for model in MODELS.keys():
        df[f'{model}_predicted'] = None
    
    # Process each question
    print("\n" + "="*100)
    print("RUNNING PREDICTIONS")
    print("="*100)
    
    for question_num in range(1, 25):
        print(f"\n{'='*100}")
        print(f"QUESTION {question_num}/24")
        print(f"{'='*100}")
        
        # Find the row for this question
        row_idx = df[df['qnum'] == question_num].index
        
        if len(row_idx) == 0:
            print(f"⚠️  No row found for qnum={question_num}, skipping...")
            continue
        
        row_idx = row_idx[0]
        
        prompt = PROMPTS[question_num]
        print(f"Prompt length: {len(prompt)} characters")
        print(f"Assigning to row {row_idx} (qnum={question_num})")
        
        # Call all 4 models
        gpt_pred = call_gpt(prompt, question_num)
        time.sleep(0.5)
        
        claude_pred = call_claude(prompt, question_num)
        time.sleep(0.5)
        
        gemini_pred = call_gemini(prompt, question_num)
        time.sleep(0.5)
        
        llama_pred = call_llama(prompt, question_num)
        time.sleep(0.5)
        
        # Store predictions in the CORRECT row
        df.loc[row_idx, 'gpt_predicted'] = gpt_pred
        df.loc[row_idx, 'claude_predicted'] = claude_pred
        df.loc[row_idx, 'gemini_predicted'] = gemini_pred
        df.loc[row_idx, 'llama_predicted'] = llama_pred
    
    # Save results
    output_file = 'pi_effects_llms_predictions.csv'
    df.to_csv(output_file, index=False)
    
    print("\n" + "="*100)
    print("✅ COMPLETED!")
    print("="*100)
    print(f"\n✓ Saved results to: {output_file}")
    print(f"✓ Total predictions: {24 * 4} (24 questions × 4 models)")
    
    # Summary statistics
    print("\n" + "="*100)
    print("PREDICTION SUMMARY")
    print("="*100)
    
    for model in MODELS.keys():
        successful = df[f'{model}_predicted'].notna().sum()
        print(f"{model.upper()}: {successful}/24 questions successfully predicted")
    
    # Show first few rows
    print("\n" + "="*100)
    print("SAMPLE OUTPUT (first 5 rows)")
    print("="*100)
    
    display_cols = ['qnum', 'gpt_predicted', 'claude_predicted', 'gemini_predicted', 'llama_predicted']
    available_cols = [col for col in display_cols if col in df.columns]
    print(df[available_cols].head().to_string(index=False))

if __name__ == "__main__":
    main()

/root/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
LLM INFERENCE - HARDCODED PROMPTS FROM PDF (FIXED VERSION)

✓ Loaded 24 rows from CSV

⚠️  WARNING: 'qnum' column not found in CSV!
   Assuming rows are in order (row 0 = Q1, row 1 = Q2, etc.)
✓ Using models: ['gpt', 'claude', 'gemini', 'llama']
✓ Hardcoded prompts: 24

RUNNING PREDICTIONS

QUESTION 1/24
Prompt length: 784 characters
Assigning to row 0 (qnum=1)
  ✓ GPT Q1: 75.0%
  ✓ CLAUDE Q1: 52.5%
  ❌ Gemini Error Q1: 429 Resource has been exhausted (e.g. check quota).
  ✓ LLAMA Q1: 47.8%

QUESTION 2/24
Prompt length: 1102 characters
Assigning to row 1 (qnum=2)
  ✓ GPT Q2: 65.0%
  ✓ CLAUDE Q2: 45.2%
  ❌ Gemini Error Q2: Invalid operation: The `response.text` quick accessor requires the response to contain a valid `Part`, but none were returned

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>